# Real Estate Amenities Deployment Project

## 1. Project Introduction

The goal of this project is to learn the basic idea of **machine learning deployment** using a small real estate amenities dataset. Instead of focusing heavily on model complexity, this project will focus on the full workflow of turning a simple machine learning model into an application that can receive input and return predictions.

The dataset contains property-level amenity information, including whether a property has a **hot tub** and whether it has a **pool**. For this project, we will build a simple model that predicts whether a property has a pool based on available features.

This project is intentionally kept small and lightweight so that the main focus can be on deployment. The planned workflow is:

1. Introduce the project and dataset
2. Prepare the data
3. Train a simple machine learning model
4. Save the trained model
5. Build a small API using FastAPI
6. Create basic deployment files
7. Deploy the API using a free/low-cost platform
8. Test the deployed model with sample inputs

By the end of this project, the final result will not just be a model inside a notebook, but a simple deployed prediction service that can be accessed through an API.

# 2. Data Preparation

Before building a machine learning model, we need to load and inspect the dataset.

Since the primary goal of this project is learning deployment, only minimal preprocessing is performed. The objective of this stage is to:

- Load the dataset
- Verify the structure of the data
- Check for missing values
- Create a numerical month feature for modeling

No extensive cleaning or exploratory analysis will be conducted.

In [4]:
# Load the data
import pandas as pd

df = pd.read_csv("DuAnKhoaHocDuLieu45.csv", sep=";")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (49311, 4)


,unified_id,month,hot_tub,pool
0,AIR10052559,2022-12,1,0
1,AIR10178668,2022-12,0,0
2,AIR10211700,2022-12,0,0
3,AIR10344705,2022-12,0,0
4,AIR10424683,2022-12,0,1


In [5]:
# Inspection
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49311 entries, 0 to 49310
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   unified_id  49311 non-null  object
 1   month       49311 non-null  object
 2   hot_tub     49311 non-null  int64 
 3   pool        49311 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 1.5+ MB


In [6]:
# Missing values
df.isnull().sum()

unified_id    0
month         0
hot_tub       0
pool          0
dtype: int64

In [7]:
# Numeric month
df["month"] = pd.to_datetime(df["month"])

df["month_num"] = df["month"].dt.month

df[["month", "month_num"]].head()

,month,month_num
0,2022-12-01,12
1,2022-12-01,12
2,2022-12-01,12
3,2022-12-01,12
4,2022-12-01,12


In [8]:
# Final preview
df.head()

,unified_id,month,hot_tub,pool,month_num
0,AIR10052559,2022-12-01,1,0,12
1,AIR10178668,2022-12-01,0,0,12
2,AIR10211700,2022-12-01,0,0,12
3,AIR10344705,2022-12-01,0,0,12
4,AIR10424683,2022-12-01,0,1,12


# 3. Model Development

The objective of this stage is to train a simple machine learning model that can be deployed later as a prediction service.

Because the primary focus of this project is deployment rather than model optimization, a basic Logistic Regression classifier will be used. The model will predict whether a property has a pool (`pool`) based on:

- `hot_tub`
- `month_num`

The dataset will be split into training and testing subsets to evaluate model performance.

In [10]:
# Feature and target
X = df[["hot_tub", "month_num"]]

y = df["pool"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (49311, 2)
Target shape: (49311,)


In [11]:
# Train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:11: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.sparse import csr_matrix, issparse


Training samples: 39448
Testing samples: 9863


In [12]:
# Train
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [13]:
# Evaluate
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9213


In [14]:
# Prediction
sample = [[1, 12]]

prediction = model.predict(sample)

print("Prediction:", prediction[0])

Prediction: 0


/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


# 4. Saving the Trained Model

After training the machine learning model, it must be saved so that it can be reused without retraining.

Model persistence allows external applications, such as web APIs, to load the trained model and generate predictions for new inputs. In this project, the trained Logistic Regression model will be stored using the Joblib library.

In [16]:
# Save model
import joblib

joblib.dump(model, "pool_predictor.pkl")

print("Model saved successfully.")

Model saved successfully.


In [17]:
# Verify
loaded_model = joblib.load("pool_predictor.pkl")

prediction = loaded_model.predict([[1, 12]])

print("Loaded model prediction:", prediction[0])

Loaded model prediction: 0


/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [18]:
# Check
import os

print(os.path.exists("pool_predictor.pkl"))

True


# 5. Building a FastAPI Prediction API

In this stage, the trained machine learning model is connected to a simple web API using FastAPI.

The API will:

- Load the saved model file
- Accept user input in JSON format
- Generate a prediction
- Return the prediction as a JSON response

This step is important because it turns the machine learning model from a notebook object into a usable service.

# 6. Preparing Files for Deployment

Before deploying the FastAPI application online, the project files must be organized so that a cloud platform can understand how to run the app.

Since this project is focused on a fast and free deployment workflow, Docker will be skipped. Instead, the deployment will use a simple file-based setup with:

- `app.py`: the FastAPI application
- `pool_predictor.pkl`: the saved machine learning model
- `requirements.txt`: the Python packages needed to run the app

This approach is beginner-friendly and works well for free deployment platforms such as Render.

# 7. Deployment Configuration

The final stage before deployment is configuring the application for cloud hosting.

A free cloud platform such as Render can be used to host the FastAPI application. The deployment process requires connecting the GitHub repository, installing project dependencies, and specifying the command used to start the API service.

Although a live deployment was not completed as part of this project, all required deployment files and configurations were successfully prepared. The application is deployment-ready and can be hosted on a cloud platform with minimal additional setup.

# 8. Testing and Reflection

After training the model and building the FastAPI application, the API was tested locally using the automatically generated Swagger documentation interface.

A sample request was submitted to the `/predict` endpoint, and the API successfully returned a prediction generated by the trained machine learning model. This confirmed that the model was correctly loaded by the application and could be used to process new inputs.

The project demonstrates the complete workflow of a deployment-focused machine learning application, including data preparation, model training, model persistence, API development, and deployment preparation.